In [9]:
# -*- coding: utf-8 -*-
"""Sales Bot - LangChain Agent con Mistral + RAG (VERSIÓN LANGCHAIN 1.3.1)"""

# ================================================================
# CELDA 1 - Instalaciones (COMPLETAS)
# ================================================================
!pip install langchain langchain-community langchain-experimental langchain-mistralai pandas openpyxl thefuzz python-Levenshtein mistralai faiss-cpu sentence-transformers --quiet
print('✅ Dependencias instaladas')

# ================================================================
# CELDA 2 - Importaciones (VERSIÓN LANGCHAIN 1.3.1)
# ================================================================
import pandas as pd
import numpy as np
import re
import unicodedata
from thefuzz import fuzz, process
from google.colab import files, userdata

# LangChain imports para versión 1.3.1
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_mistralai import ChatMistralAI

# Importaciones correctas para LangChain 1.3.1
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas')
print(f'📦 Versión de LangChain: 1.3.1')

# ================================================================
# CELDA 3 - Funciones de normalización
# ================================================================
def quitar_tildes(texto):
    texto = unicodedata.normalize('NFKD', str(texto))
    return texto.encode('ascii', 'ignore').decode('ascii')

def limpiar_texto(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def normalizar_texto_lower(texto):
    if pd.isna(texto):
        return np.nan
    texto = quitar_tildes(str(texto))
    texto = texto.lower()
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def normalizar_nombre_columna(nombre):
    nombre = quitar_tildes(str(nombre))
    nombre = nombre.lower()
    nombre = re.sub(r'[\s\-]+', '_', nombre)
    nombre = re.sub(r'[^\w]', '', nombre)
    return nombre

def normalizar_fecha(fecha):
    if pd.isna(fecha):
        return np.nan
    try:
        return pd.to_datetime(fecha, format='%m/%d/%Y %H:%M', errors='coerce')
    except:
        return pd.to_datetime(fecha, errors='coerce')

def normalizar_telefono(telefono):
    if pd.isna(telefono):
        return np.nan
    telefono = re.sub(r'[^+\d]', '', str(telefono))
    return telefono if telefono else np.nan

print('✅ Funciones de normalización cargadas')

# ================================================================
# CELDA 4 - Cargar datos
# ================================================================
print("\n📂 Por favor, sube tu archivo CSV:")
uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]

# Cargar con encoding correcto
try:
    df = pd.read_csv(nombre_archivo, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(nombre_archivo, encoding='latin-1')
    print('⚠ Archivo leído con latin-1')

print(f'✅ Archivo cargado: {df.shape[0]} filas, {df.shape[1]} columnas')

# ================================================================
# CELDA 5 - Normalizar columnas
# ================================================================
df.columns = [normalizar_nombre_columna(col) for col in df.columns]
print('✅ Columnas normalizadas a snake_case')

# ================================================================
# CELDA 6 - Limpieza general
# ================================================================
df.dropna(how='all', inplace=True)
df.drop_duplicates(inplace=True)

if 'orderdate' in df.columns:
    df['orderdate'] = df['orderdate'].apply(normalizar_fecha)
    df['order_year_month'] = df['orderdate'].dt.strftime('%Y-%m')
    df['year_id'] = df['orderdate'].dt.year

if 'phone' in df.columns:
    df['phone'] = df['phone'].apply(normalizar_telefono)

columnas_nombre = ['customername', 'contactlastname', 'contactfirstname',
                   'addressline1', 'city', 'state', 'country']
for col in columnas_nombre:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_texto)

columnas_categoricas = ['productline', 'dealsize', 'status', 'territory']
for col in columnas_categoricas:
    if col in df.columns:
        df[col] = df[col].apply(normalizar_texto_lower)

print('✅ Limpieza general completada')

# ================================================================
# CELDA 7 - Manejo de nulos
# ================================================================
for col in df.columns:
    if df[col].isnull().mean() > 0.5:
        df.drop(columns=[col], inplace=True)
        print(f'🗑 Columna eliminada: {col}')

for col in ['state', 'postalcode', 'territory']:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

if 'customername' in df.columns and 'country' in df.columns and 'sales' in df.columns:
    df.dropna(subset=['customername', 'country', 'sales'], inplace=True)

print(f'✅ Dataset después de limpieza: {df.shape[0]} filas, {df.shape[1]} columnas')

# ================================================================
# CELDA 8 - Fuzzy matching
# ================================================================
UMBRAL_SIMILITUD = 85

def deduplicar_columna(serie, umbral):
    valores_unicos = serie.dropna().unique().tolist()
    mapa = {}
    procesados = set()

    for valor in valores_unicos:
        if valor in procesados:
            continue
        similares = process.extract(valor, valores_unicos, scorer=fuzz.token_sort_ratio, limit=50)
        grupo = [v for v, score in similares if score >= umbral]
        frecuencias = serie[serie.isin(grupo)].value_counts()
        canonico = frecuencias.idxmax()
        for v in grupo:
            mapa[v] = canonico
            procesados.add(v)

    return serie.map(lambda x: mapa.get(x, x))

if 'customername' in df.columns and df.shape[0] < 10000:
    antes = df['customername'].nunique()
    df['customername'] = deduplicar_columna(df['customername'], UMBRAL_SIMILITUD)
    despues = df['customername'].nunique()
    print(f'✅ Clientes unificados: {antes} → {despues}')

# ================================================================
# CELDA 9 - Feature engineering
# ================================================================
if 'dealsize' in df.columns:
    df['is_large_deal'] = (df['dealsize'] == 'large').astype(int)

if all(col in df.columns for col in ['sales', 'quantityordered', 'msrp']):
    df['profit_margin'] = np.where(
        df['sales'] > 0,
        (df['sales'] - df['quantityordered'] * df['msrp']) / df['sales'],
        0
    ).round(4)

print('✅ Feature engineering completado')

# ================================================================
# CELDA 10 - Configurar API Key
# ================================================================
print("\n🔑 Configurando API key de Mistral...")

try:
    MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
    if MISTRAL_API_KEY:
        print('✅ API key cargada desde Colab Secrets')
    else:
        raise ValueError("Secret vacío")
except:
    MISTRAL_API_KEY = input('📝 Pegá tu MISTRAL_API_KEY: ').strip()
    if MISTRAL_API_KEY:
        print('✅ API key configurada manualmente')
    else:
        raise ValueError("❌ No se proporcionó API key")

# ================================================================
# CELDA 11 - CONSTRUCCIÓN DEL SISTEMA RAG
# ================================================================
print("\n🔄 Construyendo sistema RAG para búsqueda semántica...")

def crear_vectorstore_rag(df, columnas_texto=None):
    """
    Crea un vector store con los datos del DataFrame para búsqueda semántica.
    """
    if columnas_texto is None:
        columnas_texto = ['customername', 'country', 'city', 'productline', 'status']
        columnas_texto = [col for col in columnas_texto if col in df.columns]

    print(f"📝 Creando documentos desde {len(columnas_texto)} columnas...")

    # Crear documentos para vectorización
    documentos = []
    for idx, row in df.iterrows():
        texto_parts = []
        for col in columnas_texto:
            if pd.notna(row[col]):
                texto_parts.append(f"{col}: {row[col]}")

        if 'sales' in df.columns and pd.notna(row['sales']):
            texto_parts.append(f"ventas: ${row['sales']:,.2f}")
        if 'quantityordered' in df.columns and pd.notna(row['quantityordered']):
            texto_parts.append(f"cantidad: {row['quantityordered']}")

        texto_completo = " | ".join(texto_parts)

        # Crear documento usando langchain_core.documents.Document
        doc = Document(
            page_content=texto_completo,
            metadata={
                'index': idx,
                'customername': str(row.get('customername', '')),
                'country': str(row.get('country', '')),
                'sales': float(row.get('sales', 0))
            }
        )
        documentos.append(doc)

    print(f"✅ {len(documentos)} documentos creados")

    # Crear embeddings en español
    print("🔄 Generando embeddings con modelo multilingual...")
    try:
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
    except Exception as e:
        print(f"⚠ Error cargando modelo multilingual: {e}")
        print("Intentando con modelo base...")
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            model_kwargs={'device': 'cpu'}
        )

    # Crear vector store
    print("🔄 Indexando documentos en FAISS...")
    vectorstore = FAISS.from_documents(documentos, embeddings)

    print(f"✅ Vector store creado con {vectorstore.index.ntotal} vectores")
    return vectorstore

def buscar_contexto_relevante(vectorstore, pregunta, k=5):
    """Busca los documentos más relevantes para una pregunta."""
    if vectorstore is None:
        return "", []

    resultados = vectorstore.similarity_search_with_score(pregunta, k=k)

    contexto_parts = []
    metadatos = []

    for i, (doc, score) in enumerate(resultados):
        contexto_parts.append(f"[Documento {i+1}] (relevancia: {score:.2f})\n{doc.page_content}")
        metadatos.append(doc.metadata)

    contexto = "\n\n".join(contexto_parts)
    return contexto, metadatos

# Crear vector store para RAG (solo si el dataset no es enorme)
rag_vectorstore = None
USAR_RAG = df.shape[0] <= 5000

if USAR_RAG:
    try:
        rag_vectorstore = crear_vectorstore_rag(df)
        print("✅ Sistema RAG activado para búsqueda semántica")
    except Exception as e:
        print(f"⚠ Error creando RAG: {e}")
        print("Continuando sin RAG...")
else:
    print(f"⚠ Dataset muy grande ({df.shape[0]} filas). RAG desactivado por rendimiento.")

# ================================================================
# CELDA 12 - Crear el AGENTE de LangChain
# ================================================================
print("\n🔄 Creando agente de LangChain...")

# Configurar el LLM
llm = ChatMistralAI(
    api_key=MISTRAL_API_KEY,
    model="mistral-small-latest",
    temperature=0,
    max_retries=2,
    timeout=60
)

# Preparar estadísticas del dataset
stats = ""
if 'sales' in df.columns:
    stats += f"- Ventas totales: ${df['sales'].sum():,.2f}\n"
    stats += f"- Venta promedio: ${df['sales'].mean():,.2f}\n"
if 'country' in df.columns:
    stats += f"- Países: {df['country'].nunique()}\n"
if 'customername' in df.columns:
    stats += f"- Clientes únicos: {df['customername'].nunique():,}\n"

# Definir el prompt
prefix = f"""
Eres un analista de datos experto en ventas. Tenés acceso a un DataFrame 'df'.

📊 INFORMACIÓN DEL DATASET:
- Filas: {df.shape[0]:,}
- Columnas: {', '.join(df.columns.tolist())}

📈 ESTADÍSTICAS:
{stats}

🎯 PODÉS escribir código Python para:
- Filtrar: df[df['country'] == 'USA']
- Agrupar: df.groupby('productline')['sales'].sum()
- Calcular: df['sales'].mean(), .sum(), .max()
- Ordenar: df.sort_values('sales', ascending=False).head(10)

📋 REGLAS:
1. Usá SIEMPRE el DataFrame 'df' para cálculos exactos
2. Mostrá el código que usaste
3. Respondé en español
4. Formateá números con separadores de miles
"""

# Crear el agente
try:
    agent_executor = create_pandas_dataframe_agent(
        llm,
        df,
        prefix=prefix,
        verbose=True,
        allow_dangerous_code=True,
        handle_parsing_errors=True,
        max_iterations=3
    )
    print('\n✅ Agente creado exitosamente')
except Exception as e:
    print(f'\n⚠ Error: {e}')
    agent_executor = create_pandas_dataframe_agent(
        llm,
        df,
        verbose=True,
        allow_dangerous_code=True
    )
    print('✅ Agente creado con configuración por defecto')

print('\n' + '='*60)
print('🎉 AGENTE LISTO PARA USAR')
if USAR_RAG and rag_vectorstore:
    print('🔍 SISTEMA RAG: ACTIVADO (búsqueda semántica)')
else:
    print('⚠ SISTEMA RAG: DESACTIVADO')
print('='*60)
print(f'📊 Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print('='*60)

# ================================================================
# CELDA 13 - Función de consulta con RAG
# ================================================================
def consultar(pregunta, usar_rag=True):
    """
    Consulta al agente con opción de usar RAG para contexto adicional.
    """
    try:
        # Si RAG está activado, buscar contexto relevante
        if usar_rag and rag_vectorstore is not None:
            contexto_rag, metadatos = buscar_contexto_relevante(rag_vectorstore, pregunta, k=3)

            if contexto_rag:
                # Enriquecer la pregunta con contexto RAG
                pregunta_enriquecida = f"""
                Contexto adicional relevante (búsqueda semántica):
                {contexto_rag}

                Pregunta original: {pregunta}

                IMPORTANTE: Usa el DataFrame 'df' para cálculos exactos.
                El contexto es solo informativo para entender mejor la consulta.
                """
                resultado = agent_executor.invoke({"input": pregunta_enriquecida})
            else:
                resultado = agent_executor.invoke({"input": pregunta})
        else:
            resultado = agent_executor.invoke({"input": pregunta})

        return resultado['output']

    except Exception as e:
        return f"❌ Error: {str(e)}"

print('✅ Función consultar() con soporte RAG lista')

# ================================================================
# CELDA 14 - Bot interactivo con control RAG
# ================================================================
print('\n' + '='*55)
print('   🤖 SALES BOT - LANGCHAIN + MISTRAL + RAG')
print('='*55)
print('  ✅ Bot con búsqueda semántica (RAG) disponible')
print('')
print('  📝 Ejemplos de preguntas:')
print('     • "¿Cuál es el país con más ventas?"')
print('     • "Calcula el top 5 de clientes por ventas"')
print('     • "¿Cuál es el promedio de ventas por país?"')
print('')
print('  🛑 Comandos:')
print('     • salir   - Terminar el bot')
print('     • info    - Ver información del dataset')
print('     • rag_on  - Activar búsqueda semántica')
print('     • rag_off - Desactivar búsqueda semántica')
print('     • estado  - Ver estado del sistema')
print('='*55)
print()

# Variable para controlar modo RAG
rag_activado = USAR_RAG and rag_vectorstore is not None

while True:
    try:
        entrada = input('\n👤 Tú: ').strip()
    except (KeyboardInterrupt, EOFError):
        print('\n👋 ¡Hasta luego!')
        break

    if not entrada:
        continue

    # Comandos especiales
    if entrada.lower() == 'salir':
        print('👋 ¡Hasta luego!')
        break

    elif entrada.lower() == 'info':
        print(f'\n📊 Información del dataset:')
        print(f'   • Filas: {df.shape[0]:,}')
        print(f'   • Columnas: {df.shape[1]}')
        print(f'   • Columnas: {", ".join(df.columns[:10])}...')
        if 'sales' in df.columns:
            print(f'   • 💰 Ventas totales: ${df["sales"].sum():,.2f}')
            print(f'   • 📊 Venta promedio: ${df["sales"].mean():,.2f}')
        if 'country' in df.columns:
            print(f'   • 🌎 Países: {df["country"].nunique()}')
        if 'customername' in df.columns:
            print(f'   • 👥 Clientes: {df["customername"].nunique():,}')
        print(f'   • 🔍 RAG: {"ACTIVADO" if rag_activado else "DESACTIVADO"}')
        print()
        continue

    elif entrada.lower() == 'rag_on':
        if rag_vectorstore is not None:
            rag_activado = True
            print('✅ RAG activado - Usando búsqueda semántica')
        else:
            print('⚠ RAG no disponible (vector store no creado)')
        continue

    elif entrada.lower() == 'rag_off':
        rag_activado = False
        print('⚠ RAG desactivado - Usando solo el agente')
        continue

    elif entrada.lower() == 'estado':
        print(f'\n📊 Estado del sistema:')
        print(f'   • RAG disponible: {"Sí" if rag_vectorstore else "No"}')
        print(f'   • RAG activado: {"Sí" if rag_activado else "No"}')
        print(f'   • Filas en dataset: {df.shape[0]:,}')
        print(f'   • Columnas: {df.shape[1]}')
        continue

    # Consulta normal
    print('  ⏳ Analizando...')
    modo_texto = "🔍 [RAG]" if rag_activado else "⚡ [Directo]"
    print(f'  {modo_texto} Procesando consulta...')

    respuesta = consultar(entrada, usar_rag=rag_activado)
    print(f'\n🤖 {respuesta}\n')

# ================================================================
# CELDA 15 - Prueba rápida
# ================================================================
print("\n" + "="*55)
print("  PRUEBA RÁPIDA DEL SISTEMA")
print("="*55)

preguntas_test = [
    "¿Cuántas filas tiene el dataset?",
    "¿Cuál es el total de ventas?",
    "¿Cuáles son los nombres de las columnas?"
]

for i, pregunta in enumerate(preguntas_test, 1):
    print(f"\n❓ Pregunta {i}: {pregunta}")
    print("-" * 40)
    try:
        respuesta = consultar(pregunta, usar_rag=False)
        print(f"✅ {respuesta[:200]}...")
    except Exception as e:
        print(f"⚠ Error: {e}")

print("\n🎉 Sistema listo para usar!")

✅ Dependencias instaladas
✅ Librerías cargadas
📦 Versión de LangChain: 1.3.1
✅ Funciones de normalización cargadas

📂 Por favor, sube tu archivo CSV:


Saving sales_data_sample.csv to sales_data_sample (4).csv
⚠ Archivo leído con latin-1
✅ Archivo cargado: 2823 filas, 25 columnas
✅ Columnas normalizadas a snake_case
✅ Limpieza general completada
🗑 Columna eliminada: addressline2
🗑 Columna eliminada: state
✅ Dataset después de limpieza: 2823 filas, 24 columnas
✅ Clientes unificados: 92 → 91
✅ Feature engineering completado

🔑 Configurando API key de Mistral...
✅ API key cargada desde Colab Secrets

🔄 Construyendo sistema RAG para búsqueda semántica...
📝 Creando documentos desde 5 columnas...
✅ 2823 documentos creados
🔄 Generando embeddings con modelo multilingual...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

👋 ¡Hasta luego!

  PRUEBA RÁPIDA DEL SISTEMA

❓ Pregunta 1: ¿Cuántas filas tiene el dataset?
----------------------------------------


> Entering new AgentExecutor chain...


✅ ❌ Error: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Parsing LLM o...

❓ Pregunta 2: ¿Cuál es el total de ventas?
----------------------------------------


> Entering new AgentExecutor chain...


Thought: Necesito calcular el total de ventas del DataFrame 'df' usando la columna 'sales'. Esto se puede hacer con el método .sum() aplicado a la columna 'sales'.
Action: python_repl_ast
Action Input: df['sales'].sum()10032628.85

I now know the final answer

Final Answer: El total de ventas es **$10.032.628,85**.

> Finished chain.
✅ El total de ventas es **$10.032.628,85**....

❓ Pregunta 3: ¿Cuáles son los nombres de las columnas?
----------------------------------------


> Entering new AgentExecutor chain...


Thought: Necesito obtener los nombres de las columnas del DataFrame 'df' para responder la pregunta.
Action: python_repl_ast
Action Input: df.columns.tolist()['ordernumber', 'quantityordered', 'priceeach', 'orderlinenumber', 'sales', 'orderdate', 'status', 'qtr_id', 'month_id', 'year_id', 'productline', 'msrp', 'productcode', 'customername', 'phone', 'addressline1', 'city', 'postalcode', 'country', 'territory', 'contactlastname', 'contactfirstname', 'dealsize', 'order_year_month', 'is_large_deal', 'profit_margin']I now know the final answer

Final Answer:
Las columnas del DataFrame 'df' son:

1. ordernumber
2. quantityordered
3. priceeach
4. orderlinenumber
5. sales
6. orderdate
7. status
8. qtr_id
9. month_id
10. year_id
11. productline
12. msrp
13. productcode
14. customername
15. phone
16. addressline1
17. city
18. postalcode
19. country
20. territory
21. contactlastname
22. contactfirstname
23. dealsize
24. order_year_month
25. is_large_deal
26. profit_margin

> Finished chain.
✅ L